# [PBT] 화장품 이커머스 ㅣ 04. 설명변수 생성

## 이 노트북이 하는 일

03번에서 만든 대상 테이블(1행 = 1고객)에 **설명변수 X를 붙인다.**

| 조각 | 내용 | 담당 | 상태 |
|---|---|---|---|
| ① | 대상 고객과 t₀ 확정 | 배지환 | ✅ 완료 (`02a_target.csv`) |
| ② | 목표변수 Y 생성 | 이슬기 | 진행 중 |
| **③** | **설명변수 X 생성** | 배지환 | ← **이 노트북** |

### 만들 변수

| 변수 | 출처 | 척도 |
|---|---|---|
| `price` · `brand` · `category_id` | t₀ 행에 이미 있음 (①에서 확보) | 연속형 / 명목형 |
| `cart_timeband` · `cart_weekday` | **t₀에서 파생** | 명목형 |
| `has_prior_view` | **t₀ 직전 30분 창** | 이진 |
| `prior_view_cnt` | 〃 | 연속형 |
| `prior_uniq_product_cnt` | 〃 | 연속형 |
| `cart_product_view_cnt` | 〃 | 연속형 |

### 30분 창은 t₀의 왼쪽이다

```
        ←──── 30분 ────→ │  t₀  │ ←────────── 7일 ──────────→

         이 노트북(X)      담기    이슬기(Y)
```

t₀ 이후 정보를 X에 넣으면 **결과를 미리 아는 것**이 되므로, 반드시 왼쪽만 본다.

경계는 **`t₀ − 30분 ≤ 조회시각 ≤ t₀`**로 양끝을 포함한다.
같은 초의 조회를 포함하는 이유는 미래 정보가 아니라 동시 정보이기 때문이며,

초 단위 기록이라 "보고 바로 담기"가 같은 초에 찍히는 경우가 흔하기 때문이다.

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)             # 폰트 경로가 상대경로라 여기서 실행해야 한다
sys.path.insert(0, BASE)

from pandas import read_parquet, read_csv, to_datetime
from pandas import DataFrame, Timedelta, cut, set_option

from helpers import my_qtcheck

set_option("display.max_columns", 50)

WINDOW = 30      # 조회 변수 구간 (분). 8/26에 확정, 이후 변경하지 않는다

### 2. 원본 로그 불러오기

In [2]:

OUT = BASE + r"\output"

df = read_parquet(OUT + r"\01_qtcheck.parquet")
df = my_qtcheck.set_type(df, as_category=["category_id"])

<class 'pandas.DataFrame'>
RangeIndex: 8259159 entries, 0 to 8259158
Data columns (total 9 columns):
 #   Column         Dtype         
---  ------         -----         
 0   event_time     datetime64[us]
 1   event_type     category      
 2   product_id     int32         
 3   category_id    category      
 4   category_code  category      
 5   brand          category      
 6   price          float64       
 7   user_id        int64         
 8   user_session   str           
dtypes: category(4), datetime64[us](1), float64(1), int32(1), int64(1), str(1)
memory usage: 615.3 MB


### 3. 대상 테이블 불러오기

CSV는 타입을 보존하지 않으므로 날짜와 명목형을 다시 변환한다.

In [3]:
target = read_csv(OUT + r"\02a_target.csv")

target["t0"] = to_datetime(target["t0"], format="%Y-%m-%d %H:%M:%S")
target = my_qtcheck.set_type(target, as_category=["brand", "category_id"])

print(f"대상 고객 : {len(target):,}명")
target.head()

<class 'pandas.DataFrame'>
RangeIndex: 108458 entries, 0 to 108457
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   user_id      108458 non-null  int64         
 1   t0           108458 non-null  datetime64[us]
 2   product_id   108458 non-null  int64         
 3   price        108458 non-null  float64       
 4   brand        66322 non-null   category      
 5   category_id  108458 non-null  category      
dtypes: category(2), datetime64[us](1), float64(1), int64(2)
memory usage: 3.7 MB
대상 고객 : 108,458명


,user_id,t0,product_id,price,brand,category_id
0,543079411,2019-10-11 00:00:17,5591314,4.29,NaN,1487580005268456287
1,555391563,2019-10-11 00:01:47,5816172,5.24,grattol,1602943681873052386
2,385999154,2019-10-11 00:01:51,5753492,13.02,italwax,1487580013069861041
3,558950685,2019-10-11 00:02:04,5875777,2.05,NaN,2084144451428549153
4,558519615,2019-10-11 00:02:17,5810157,1.43,irisk,1487580009445982239


## #02. view 이벤트에 t₀ 붙이기

### 왜 병합이 먼저인가

"30분"이라는 규칙은 모두에게 같지만 **t₀가 고객마다 다르므로 실제 시간 구간은 19만 개**다.

| 고객 | t₀ | 세어야 할 구간 |
|---|---|---|
| A | 10/03 14:20:11 | 13:50:11 ~ 14:20:11 |
| B | 11/20 22:05:43 | 21:35:43 ~ 22:05:43 |

`groupby("user_id")`는 고객별로 묶으라는 명령이지 **고객별로 다른 시간 조건을 적용하라**는 명령이 아니다.
게다가 view 행 자체에는 그 고객의 t₀가 없으므로, 행 하나만 보아서는 집계 대상인지 판정할 수 없다.

그래서 **각 view 행 옆에 그 고객의 t₀를 먼저 적어놓는다.** 그것이 병합이다.

| | user_id | event_time | product_id | t0 | cart_product |
|---|---|---|---|---|---|
| 병합 전 | 5001 | 14:15 | 30 | — | — |
| 병합 후 | 5001 | 14:15 | 30 | **14:20** | **30** |

병합 후에는 그 행 **혼자서** 판정이 된다.

### 1. view 이벤트 추출

In [4]:
views = df.loc[df["event_type"] == "view", ["user_id", "event_time", "product_id"]]
print(f"view 이벤트 : {len(views):,}행")

view 이벤트 : 3,922,684행


### 2. 대상 고객의 t₀를 붙인다

- 대상 테이블의 `product_id`는 **담은 상품**이므로 `cart_product`로 이름을 바꾼다.
  그대로 두면 view의 `product_id`와 이름이 충돌한다.
- 안쪽 병합이므로 **대상 고객이 아닌 사람의 view는 자동으로 빠진다.**

In [5]:
key = target[["user_id", "t0", "product_id"]].rename(
          columns={"product_id": "cart_product"})

v = views.merge(key, on="user_id")

print(f"대상 고객의 view : {len(v):,}행")
v.head()

대상 고객의 view : 1,608,194행


,user_id,event_time,product_id,t0,cart_product
0,485644718,2019-10-01 03:04:42,5550686,2019-11-04 18:52:31,89756
1,555448260,2019-10-01 03:06:35,5800917,2019-11-14 00:24:22,5783987
2,555448260,2019-10-01 03:09:58,5890122,2019-11-14 00:24:22,5783987
3,533222198,2019-10-01 03:11:16,5723490,2019-10-29 21:19:38,5896788
4,441342983,2019-10-01 03:15:26,5595916,2019-10-12 04:17:44,5864810


## #03. 30분 창으로 자르기

경계는 양끝 포함이다. `t₀ − 30분 ≤ 조회시각 ≤ t₀`

In [6]:
lo = v["t0"] - Timedelta(minutes=WINDOW)

v = v.loc[(v["event_time"] >= lo) & (v["event_time"] <= v["t0"])]

print(f"창 안의 view : {len(v):,}행")
print(f"조회가 있었던 고객 : {v['user_id'].nunique():,}명")

창 안의 view : 138,803행
조회가 있었던 고객 : 60,001명


## #04. 집계

`user_id`로 묶어 세 가지를 센다.

| 변수 | 세는 방법 |
|---|---|
| `prior_view_cnt` | 행 수 |
| `prior_uniq_product_cnt` | `product_id`의 고유값 수 |
| `cart_product_view_cnt` | `product_id`가 담은 상품과 같은 행 수 |

In [7]:
v["is_cart_product"] = v["product_id"] == v["cart_product"]

g = v.groupby("user_id")

agg = DataFrame({
    "prior_view_cnt"        : g.size(),
    "prior_uniq_product_cnt": g["product_id"].nunique(),
    "cart_product_view_cnt" : g["is_cart_product"].sum(),
}).reset_index()

print(f"집계된 고객 : {len(agg):,}명")
agg.head()

집계된 고객 : 60,001명


,user_id,prior_view_cnt,prior_uniq_product_cnt,cart_product_view_cnt
0,5493470,1,1,1
1,10280338,1,1,1
2,10968431,1,1,1
3,13117447,1,1,0
4,14297993,1,1,1


## #05. 대상 테이블에 붙이고 0으로 채우기

### ⚠️ 이 단계 최대의 함정

`groupby`는 **데이터가 있는 그룹만** 만든다.
30분 안에 조회가 하나도 없던 고객은 `#03` 필터에서 행이 전부 사라져
`#04` 결과에 **아예 등장하지 않는다.**

그런 고객이 **약 30%(6만 명 내외)**로 예상된다.
8/26에 측정한 `has_prior_view = 0` 비율이 30.4%였던 것이 이들이다.

**"조회가 없었다"는 결측이 아니라 0이다.** 반드시 0으로 채운다.

In [8]:
x = target.merge(agg, on="user_id", how="left")

CNT_COLS = ["prior_view_cnt", "prior_uniq_product_cnt", "cart_product_view_cnt"]

print("0으로 채우기 전 결측")
print(x[CNT_COLS].isna().sum().to_string())

for c in CNT_COLS:
    x[c] = x[c].fillna(0).astype("int32")

x["has_prior_view"] = (x["prior_view_cnt"] > 0).astype("int8")

print()
print("0으로 채운 뒤 결측 :", x[CNT_COLS].isna().sum().sum())

0으로 채우기 전 결측
prior_view_cnt            48457
prior_uniq_product_cnt    48457
cart_product_view_cnt     48457

0으로 채운 뒤 결측 : 0


## #06. t₀에서 파생하는 변수

### 1. `cart_timeband` — 담은 시간대 4구간

시각을 24개 명목값으로 두면 더미변수가 23개 생겨 계수 해석이 흩어진다.
8/26에 확정한 경계로 묶는다. **심야 0–5 / 오전 6–11 / 오후 12–17 / 저녁 18–23**

> 시간대는 UTC+3으로 보정된 값이므로 현지 시각 기준이다.

In [9]:
BINS   = [-1, 5, 11, 17, 23]
LABELS = ["심야", "오전", "오후", "저녁"]

x["cart_timeband"] = cut(x["t0"].dt.hour, bins=BINS, labels=LABELS)

tb = x["cart_timeband"].value_counts().sort_index()
DataFrame({"고객 수": tb.values,
           "비율(%)": (tb.values / len(x) * 100).round(2)}, index=tb.index)

,고객 수,비율(%)
cart_timeband,,
심야,9107,8.40
오전,23663,21.82
오후,39186,36.13
저녁,36502,33.66


### 2. `cart_weekday` — 담은 요일

주말 여부는 요일에서 그대로 읽히므로 별도 변수를 두지 않는다.

In [10]:
WD = ["월", "화", "수", "목", "금", "토", "일"]

x["cart_weekday"] = x["t0"].dt.dayofweek.map(dict(enumerate(WD))).astype("category")

wd = x["cart_weekday"].value_counts().reindex(WD)
DataFrame({"고객 수": wd.values,
           "비율(%)": (wd.values / len(x) * 100).round(2)}, index=WD)

,고객 수,비율(%)
월,15558,14.34
화,14730,13.58
수,14758,13.61
목,15955,14.71
금,18235,16.81
토,15244,14.06
일,13978,12.89


## #07. 검증

### 1. 행 수와 결측

대상 고객 수가 그대로여야 한다. 병합에서 행이 늘거나 줄면 키가 잘못된 것이다.

In [11]:
print("대상 고객 :", format(len(target), ","))
print("결과 행 수 :", format(len(x), ","))
print("일치      :", len(x) == len(target))
print("user_id 중복 :", x["user_id"].duplicated().sum())
print()
my_qtcheck.check_missing_values(x)

대상 고객 : 108,458
결과 행 수 : 108,458
일치      : True
user_id 중복 : 0



,Missing Count,Missing Ratio (%)
user_id,0,0.000000
t0,0,0.000000
product_id,0,0.000000
price,0,0.000000
brand,42136,38.850062
category_id,0,0.000000
prior_view_cnt,0,0.000000
prior_uniq_product_cnt,0,0.000000
cart_product_view_cnt,0,0.000000
has_prior_view,0,0.000000


### 2. 논리적으로 성립해야 하는 관계

세 가지가 모두 참이어야 한다. 하나라도 깨지면 집계가 잘못된 것이다.

| 관계 | 왜 |
|---|---|
| `cart_product_view_cnt ≤ prior_view_cnt` | 담은 상품 조회는 전체 조회의 일부다 |
| `prior_uniq_product_cnt ≤ prior_view_cnt` | 상품 종류 수가 조회 건수를 넘을 수 없다 |
| 조회가 있으면 종류 수도 1 이상 | 무언가를 보았다면 최소 한 종류다 |

In [12]:
c1 = (x["cart_product_view_cnt"] <= x["prior_view_cnt"]).all()
c2 = (x["prior_uniq_product_cnt"]  <= x["prior_view_cnt"]).all()
c3 = ((x["prior_view_cnt"] == 0) | (x["prior_uniq_product_cnt"] >= 1)).all()
c4 = ((x["has_prior_view"] == 1) == (x["prior_view_cnt"] > 0)).all()

print("담은 상품 조회 ≤ 전체 조회 :", c1)
print("상품 종류 수  ≤ 전체 조회 :", c2)
print("조회 있으면 종류 ≥ 1      :", c3)
print("has_prior_view 정합       :", c4)

담은 상품 조회 ≤ 전체 조회 : True
상품 종류 수  ≤ 전체 조회 : True
조회 있으면 종류 ≥ 1      : True
has_prior_view 정합       : True


### 3. `has_prior_view = 0` 비율 — 8/26 진단과 대조

8/26에 창 길이를 정할 때 측정한 값이 **30.4%**였다.
그때는 **모든 cart**를 대상으로 했고 지금은 **고객별 최초 cart**만 대상이므로 정확히 같지는 않다.
최초 담기는 그 고객의 첫 행동에 가까우므로 **0 비율이 다소 높게 나오는 것이 자연스럽다.**

그러나 대역이 크게 벗어난다면(예: 5% 또는 70%) 집계 어딘가가 잘못된 것이다.

In [13]:
zero = (x["has_prior_view"] == 0).mean()

print(f"has_prior_view = 0 : {zero:.1%}  ({(x['has_prior_view'] == 0).sum():,}명)")
print(f"has_prior_view = 1 : {1 - zero:.1%}")
print()
print("8/26 진단값 (모든 cart 기준) : 30.4%")

has_prior_view = 0 : 44.7%  (48,457명)
has_prior_view = 1 : 55.3%

8/26 진단값 (모든 cart 기준) : 30.4%


### 4. 분포 확인

조회 변수는 0이 많고 오른쪽 꼬리가 긴 형태일 것으로 예상된다.
왜도가 크면 ⑤ 전처리에서 로그변환 대상이 된다.

In [14]:
num = ["prior_view_cnt", "prior_uniq_product_cnt", "cart_product_view_cnt"]

desc = x[num].describe().T
desc["skew"] = x[num].skew()
desc["0의 비율(%)"] = [(x[c] == 0).mean() * 100 for c in num]
desc.round(3)

,count,mean,std,min,25%,50%,75%,max,skew,0의 비율(%)
prior_view_cnt,108458.0,1.280,2.345,0.0,0.0,1.0,1.0,71.0,5.901,44.678
prior_uniq_product_cnt,108458.0,1.120,1.988,0.0,0.0,1.0,1.0,71.0,6.044,44.678
cart_product_view_cnt,108458.0,0.458,0.658,0.0,0.0,0.0,1.0,10.0,1.751,61.305


### 5. 무작위 3명 수기 확인

원본 로그에서 해당 고객의 t₀ 직전 30분 기록을 직접 뽑아,
테이블의 네 변수가 손으로 세어도 같은 값인지 확인한다.

In [15]:
sample_ids = x["user_id"].sample(3, random_state=3217).to_list()

for uid in sample_ids:
    row = x.loc[x["user_id"] == uid]
    t   = row["t0"].iloc[0]

    print("=" * 78)
    print("user_id :", uid, "  |  t0 :", t, "  |  담은 상품 :", row["product_id"].iloc[0])
    print("[테이블]")
    print(row[["has_prior_view"] + num].to_string(index=False))

    log = df.loc[(df["user_id"] == uid)
                 & (df["event_time"] >= t - Timedelta(minutes=WINDOW))
                 & (df["event_time"] <= t),
                 ["event_time", "event_type", "product_id"]].sort_values("event_time")

    print(f"[원본 로그 — t0 직전 {WINDOW}분]")
    print(log.to_string(index=False) if len(log) else "  (기록 없음)")
    print()

user_id : 250311201   |  t0 : 2019-11-03 10:03:59   |  담은 상품 : 5671878
[테이블]
 has_prior_view  prior_view_cnt  prior_uniq_product_cnt  cart_product_view_cnt
              0               0                       0                      0
[원본 로그 — t0 직전 30분]
         event_time event_type  product_id
2019-11-03 10:03:59       cart     5671878

user_id : 565763975   |  t0 : 2019-10-30 15:34:21   |  담은 상품 : 5799891
[테이블]
 has_prior_view  prior_view_cnt  prior_uniq_product_cnt  cart_product_view_cnt
              1               1                       1                      1
[원본 로그 — t0 직전 30분]
         event_time event_type  product_id
2019-10-30 15:33:05       view     5799891
2019-10-30 15:34:21       cart     5799891

user_id : 560978725   |  t0 : 2019-10-16 22:00:27   |  담은 상품 : 5751383
[테이블]
 has_prior_view  prior_view_cnt  prior_uniq_product_cnt  cart_product_view_cnt
              0               0                       0                      0
[원본 로그 — t0 직전 30분]
         event_tim

> **손으로 확인하는 방법**
> - `event_type`이 `view`인 행의 수 = `prior_view_cnt`
> - 그 행들의 `product_id` 종류 수 = `prior_uniq_product_cnt`
> - 그중 담은 상품과 같은 것의 수 = `cart_product_view_cnt`
>
> 마지막 줄의 `cart` 행은 t₀ 그 자체이므로 세지 않는다.

## #08. 저장

이슬기의 Y 결과와 합쳐 마스터 테이블을 완성한다.

In [16]:
x.to_csv(OUT + r"\02b_x.csv", index=False, encoding="utf-8-sig")

print("저장 완료 :", OUT + r"\02b_x.csv")
print(f"{len(x):,}행 × {x.shape[1]}열")
x.info()

저장 완료 : .\output\02b_x.csv
108,458행 × 12열
<class 'pandas.DataFrame'>
RangeIndex: 108458 entries, 0 to 108457
Data columns (total 12 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   user_id                 108458 non-null  int64         
 1   t0                      108458 non-null  datetime64[us]
 2   product_id              108458 non-null  int64         
 3   price                   108458 non-null  float64       
 4   brand                   66322 non-null   category      
 5   category_id             108458 non-null  category      
 6   prior_view_cnt          108458 non-null  int32         
 7   prior_uniq_product_cnt  108458 non-null  int32         
 8   cart_product_view_cnt   108458 non-null  int32         
 9   has_prior_view          108458 non-null  int8          
 10  cart_timeband           108458 non-null  category      
 11  cart_weekday            108458 non-null  category      
dtyp

In [17]:
# =========================================================
# [진단] 창 길이별 조회 변수 분포 비교
#
#  30분 창이 prior_view_cnt의 변동을 눌러버린 것인지 확인한다.
#  ★ Y는 보지 않는다. X의 분포만 보는 것이므로
#    "결과를 보고 설계를 바꾸는" 반칙에 해당하지 않는다.
# =========================================================

vm = views.merge(key, on="user_id")          # 창을 자르기 전 상태로 되돌린다
vm["is_cart_product"] = vm["product_id"] == vm["cart_product"]

rows = []

for w in [10, 30, 60, 180]:
    sel = vm.loc[(vm["event_time"] >= vm["t0"] - Timedelta(minutes=w))
                 & (vm["event_time"] <= vm["t0"])]

    gg = sel.groupby("user_id")

    a = DataFrame({
        "cnt" : gg.size(),
        "uniq": gg["product_id"].nunique(),
        "same": gg["is_cart_product"].sum(),
    }).reindex(target["user_id"]).fillna(0)

    rows.append({
        "창(분)"          : w,
        "0건 (%)"         : round((a["cnt"] == 0).mean() * 100, 2),
        "1건 (%)"         : round((a["cnt"] == 1).mean() * 100, 2),
        "2건 이상 (%)"    : round((a["cnt"] >= 2).mean() * 100, 2),
        "평균"            : round(a["cnt"].mean(), 3),
        "중앙값"          : a["cnt"].median(),
        "75%"             : a["cnt"].quantile(.75),
        "90%"             : a["cnt"].quantile(.90),
        "최대"            : int(a["cnt"].max()),
        "cnt↔uniq 상관"   : round(a["cnt"].corr(a["uniq"]), 3),
        "두 값 일치 (%)"  : round((a["cnt"] == a["uniq"]).mean() * 100, 2),
        "담은상품 0회 (%)": round((a["same"] == 0).mean() * 100, 2),
        "담은상품 평균"   : round(a["same"].mean(), 3),
    })

DataFrame(rows).set_index("창(분)").T

창(분),10,30,60,180
0건 (%),46.650,44.680,44.070,43.470
1건 (%),31.290,30.670,30.480,30.200
2건 이상 (%),22.060,24.660,25.460,26.330
평균,1.063,1.280,1.358,1.439
중앙값,1.000,1.000,1.000,1.000
75%,1.000,1.000,2.000,2.000
90%,3.000,3.000,3.000,3.000
최대,45.000,71.000,127.000,223.000
cnt↔uniq 상관,0.966,0.969,0.971,0.970
두 값 일치 (%),91.890,90.000,89.370,88.680


### 해석 — 창을 넓혀도 그림이 거의 바뀌지 않는다

이 표는 "30분이라는 창이 `prior_view_cnt`의 변동을 눌러버린 건 아닌가?"를 확인하려고, 같은 집계를 **10분·30분·60분·180분** 네 가지 창 길이로 반복해 나란히 놓은 것이다. 열이 창 길이, 행이 각 지표다.

1. **0건 / 1건 / 2건 이상 (%)** — 창을 10분→180분으로 18배 늘려도 "조회가 아예 없는 고객" 비율은 46.65% → 43.47%로 3%p 남짓만 줄어든다. 창을 크게 넓혀도 무(無)조회 고객이 크게 줄지는 않는다.
2. **평균 vs 중앙값** — 평균은 1.063 → 1.439로 꾸준히 오르지만, **중앙값은 네 창 모두 1로 그대로**다. 즉 "전형적인" 고객의 행동은 창 길이와 무관하게 똑같고, 평균이 올라가는 건 소수의 극단값(최대 45건 → 223건) 때문이다 — 창을 넓히면 꼬리만 길어진다.
3. **cnt↔uniq 상관 / 두 값 일치(%)** — 조회 건수와 조회한 상품 종류 수는 창 길이와 상관없이 상관계수 0.97 안팎, 일치율 88~92%로 항상 비슷하다. 고객들이 "같은 상품을 여러 번 다시 보는" 경우가 드물다는 패턴은 창 길이가 만든 결과가 아니라 실제 행동의 특징이라는 뜻이다.
4. **담은상품 0회(%) / 담은상품 평균** — "담기 전에 그 상품을 본 적이 있는가"도 창을 넓혀도 61~62% 선에서 거의 움직이지 않는다.

**결론**: 창 길이를 18배까지 벌려도 중앙값·상관관계·비율 같은 핵심 지표는 거의 흔들리지 않는다. 즉 **30분이라는 선택이 `prior_view_cnt`의 변동성을 인위적으로 눌러버리는 것이 아니다** — 창을 넓혀서 얻는 건 대부분 평균만 끌어올리는 소수의 극단값이고, 대다수 고객의 실질적인 신호는 30분 안에서 이미 충분히 잡힌다는 근거가 된다.

---

> 여기 있던 **번인 진단 셀(코호트별 Y · 번인 시작일별 표 · 조회 변수↔Y)** 은
> `PBT_화장품이커머스_07_번인진단.ipynb` 로 옮겼다.
> 04번은 설명변수 X를 만드는 노트북이고, 그 셀들은 목표변수 Y를 읽기 때문이다.
